# VideoDB Understanding: VLM on Sandbox Compute

<a href="https://colab.research.google.com/github/video-db/videodb-cookbook/blob/preview/guides/indexing-v2/understanding/vlm/sandbox-compute.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Run a VLM analyzer against a **sandbox-compute model** instead of a third-party API.

- A **sandbox** is reserved GPU capacity on VideoDB Open Compute. You create it, wait for it to become active, run jobs on it, and stop it when you are done.
- **Sandbox-compute models** (Qwen, Gemma, and friends) run inside that sandbox, so they need a live `sandbox_id`.
- **Third-party models** (`google/gemini-2.5-flash`, `openai/gpt-4o`, and the `basic` / `pro` / `ultra` aliases) are called over the provider API and do **not** need a sandbox at all. If that is what you want, use `vlm-guide.ipynb` instead.


## 1. Install dependencies

In [ ]:
!pip install -q --force-reinstall --no-cache-dir "git+https://github.com/video-db/videodb-python.git@feat/add-indexing-v2" python-dotenv

## 2. Connect to VideoDB

In [ ]:
import json
import os
from getpass import getpass

from dotenv import load_dotenv
from videodb import connect, SandboxModel, SandboxTier

load_dotenv()
os.environ["VIDEO_DB_API_KEY"] = os.getenv("VIDEO_DB_API_KEY") or getpass("Enter your VideoDB API key: ")

conn = connect(api_key=os.environ["VIDEO_DB_API_KEY"])
collection = conn.get_collection()

print("Connected to VideoDB")
print("Collection:", collection.id)

## 3. Choose a video

By default, this notebook uploads the same sample video as the other VLM guides: **Silicon Valley - Gilfoyle is free for hire**. To use an existing video instead, comment the upload line and uncomment the `get_video` lines in the next cell.


In [ ]:
VIDEO_URL = "https://www.youtube.com/watch?v=vVlEVRKv4is"  # Silicon Valley - Gilfoyle is free for hire

video = collection.upload(VIDEO_URL)

# To use an existing video instead, comment the upload line above and uncomment these lines:
# VIDEO_ID = "m-..."
# video = collection.get_video(VIDEO_ID)

print("Collection:", collection.id)
print("Video:", video.id)
video.play()


## 4. Create a sandbox

A sandbox is a warm GPU pool dedicated to your account. Pick a tier based on the size of the model you want to run.

| Tier | Good for |
|---|---|
| `small` | Smaller VLMs (`gemma-4-E2B`, `Qwen3.5-9B`), OmniVoice |
| `medium` | Larger VLMs (`gemma-4-26B`, `gemma-4-31B`, `Qwen3.5-27B`), FLUX |

This notebook creates a `medium` sandbox so the larger VLMs are available.

> Sandbox billing is based on runtime. Stop it when you are done (see the cleanup section at the end).


In [ ]:
# Create a sandbox (returns immediately in 'provisioning' state)
sandbox = conn.create_sandbox(tier=SandboxTier.medium)
print(f"Sandbox: {sandbox.id}, Status: {sandbox.status}, Tier: {sandbox.tier}")

In [ ]:
# Wait until the sandbox is active before submitting any jobs.
sandbox.wait_for_ready(timeout=300, interval=5)
print(f"Sandbox ready: {sandbox.id}, Status: {sandbox.status}")

In [ ]:
# Manual polling alternative
sandbox.refresh()
print(f"Status: {sandbox.status}, Active: {sandbox.is_active}")

In [ ]:
# List all sandboxes on your account
for sb in conn.list_sandboxes():
    print(f"{sb.id} | {sb.name} | {sb.tier} | {sb.status}")

In [ ]:
# Fetch a specific sandbox by ID
sb = conn.get_sandbox(sandbox.id)
print(f"{sb.id} | {sb.status}")

## 5. Pick a sandbox-compute model

VideoDB exposes sandbox-compatible models through the `SandboxModel` enum. The VLM-capable ones are:

| Model enum | Model id | Minimum tier |
|---|---|---|
| `SandboxModel.GEMMA_4_E2B` | `google/gemma-4-E2B-it` | `small` |
| `SandboxModel.QWEN_9B` | `Qwen/Qwen3.5-9B` | `small` |
| `SandboxModel.GEMMA_4_26B` | `google/gemma-4-26B-A4B-it` | `medium` |
| `SandboxModel.QWEN_27B` | `Qwen/Qwen3.5-27B` | `medium` |
| `SandboxModel.GEMMA_4_31B` | `google/gemma-4-31B-it` | `medium` |

### A slash in the name does not mean sandbox compute

Compare these two:

```python
"google/gemma-4-31B-it"    # sandbox compute, runs in your sandbox
"google/gemini-2.5-flash"  # third-party, runs on Google's API
```

Both look like HuggingFace-style repo ids, but the string shape carries no routing information. **Routing is decided by looking the model up in the sandbox-compute model registry**, not by whether the name contains a `/`.

- A model served by sandbox compute → the request routes there and requires `sandbox_id`.
- A third-party model → the request goes to the provider and `sandbox_id` is not needed.


In [ ]:
SANDBOX_MODEL = SandboxModel.QWEN_27B  # "Qwen/Qwen3.5-27B"

# Other options on a medium sandbox:
# SANDBOX_MODEL = SandboxModel.GEMMA_4_31B   # "google/gemma-4-31B-it"
# SANDBOX_MODEL = SandboxModel.GEMMA_4_26B   # "google/gemma-4-26B-A4B-it"

print("Model:", SANDBOX_MODEL)
print("Sandbox:", sandbox.id)

## 6. Run the VLM analyzer on sandbox compute

The sandbox-compute path is deliberately narrow right now. Keep the request to exactly this shape:

- **Exactly one `vlm` analyzer.** Multiple VLM analyzers in one understanding are not supported on this path.
- **No `object_detection` analyzer.** Detection models are not served from the sandbox.
- **No `inputs`.** Upstream analyzer context (`inputs: ["transcript", "objects"]`, as used in `vlm-guide.ipynb`) builds per-group context, which the sandbox-compute path does not support yet — the node will reject the request.

Everything else works as usual: `sampling` controls how many frames per segment are sent, and `segmentation` controls how the video is cut into segments.


In [ ]:
VLM_ANALYZER = {
    "type": "vlm",
    "name": "scene",
    "sampling": {"strategy": "uniform", "frame_count": 8},
    "config": {
        "model": SANDBOX_MODEL,
        "sandbox_id": sandbox.id,
        "prompt": "Describe what is happening in this scene.",
    },
}

SEGMENTATION = {"type": "shot", "threshold": 30}

### Inspect the request payload before submitting

This is the exact contract the sandbox-compute VLM path expects. Useful as a reference when wiring this up from another client.


In [ ]:
request_payload = {
    "analyzers": [VLM_ANALYZER],
    "segmentation": SEGMENTATION,
}

print(json.dumps(request_payload, indent=2, default=str))

### Submit

In [ ]:
understanding = video.understand(
    analyzers=[{
        "type": "vlm",
        "name": "scene",
        "sampling": {"strategy": "uniform", "frame_count": 8},
        "config": {
            "model": SANDBOX_MODEL,
            "sandbox_id": sandbox.id,
            "prompt": "Describe what is happening in this scene.",
        },
    }],
    segmentation={"type": "shot", "threshold": 30},
)

print("Understanding:", understanding.id)

## 7. Inspect the results

Same helper shape as `vlm-guide.ipynb`: wait for the run, list the analyzers, then preview a few segments.


In [ ]:
def show_vlm_output(understanding, analyzer_name="scene", max_scenes=5):
    understanding.wait_until_complete(timeout=3600, poll_interval=15)

    print("Understanding:", understanding.id)
    print("Status:", understanding.status)
    print()

    for analyzer in understanding.list_analyzers():
        print(analyzer.name, analyzer.type, analyzer.status)

    output = understanding.get_analyzer(analyzer_name).get_output()
    scenes = output.get("scenes", output) if isinstance(output, dict) else output
    scenes = scenes or []

    print("Preview")
    print("=" * 60)
    for scene in scenes[:max_scenes]:
        print(f"{scene.get('start')}s → {scene.get('end')}s")
        print(scene.get("data"))
        print("-" * 60)

    return scenes

In [ ]:
scenes = show_vlm_output(understanding, "scene")

## 8. Cleanup

Uncomment the lines below to stop the sandbox (ending compute billing) and delete the understanding created by this notebook.


In [ ]:
# sandbox.stop()
# print(f"Sandbox {sandbox.id} status: {sandbox.status}")

# sandbox.wait_for_stop(timeout=120)
# print(f"Sandbox {sandbox.id} final status: {sandbox.status}")

In [ ]:
# understanding.delete()

## Quick reference

### Sandbox-compute VLM analyzer

```python
{
    "type": "vlm",
    "name": "scene",
    "sampling": {"strategy": "uniform", "frame_count": 8},
    "config": {
        "model": SandboxModel.QWEN_27B,
        "sandbox_id": sandbox.id,
        "prompt": "Describe what is happening in this scene.",
    },
}
```

### Third-party VLM analyzer (no sandbox)

```python
{
    "type": "vlm",
    "name": "scene",
    "sampling": {"strategy": "uniform", "frame_count": 8},
    "config": {
        "model": "google/gemini-2.5-flash",
        "prompt": "Describe what is happening in this scene.",
    },
}
```

### Not supported on the sandbox-compute path (yet)

- `inputs: [...]` on the VLM analyzer
- an `object_detection` analyzer in the same request
- more than one `vlm` analyzer in the same request
